# Project Management Analytics

This notebook demonstrates exploratory data analysis and predictive modeling on a synthetic dataset of project management metrics. The goal is to illustrate analysis techniques relevant for business analysts, program managers, and data analysts.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, r2_score

# Set plotting style
sns.set(style='whitegrid')
%matplotlib inline


In [ ]:
# Load synthetic dataset
df = pd.read_csv('dataset.csv', parse_dates=['start_date','end_date'])

# Preview dataset
print('Dataset shape:', df.shape)
df.head()


In [ ]:
# Summary statistics
print('Dataset information:')
df.info()

print('
Summary statistics:')
df.describe(include='all')

# Duration distribution
plt.figure(figsize=(8,4))
sns.histplot(df['duration_days'], bins=20, kde=True)
plt.title('Project Duration Distribution')
plt.xlabel('Duration (days)')
plt.ylabel('Count')
plt.show()


In [ ]:
# Convert budget to millions for readability
budget_m = df['budget_k']

# Scatter plot: Budget vs Risk Score colored by Success
plt.figure(figsize=(8,5))
sns.scatterplot(x=budget_m, y=df['risk_score'], hue=df['success'], palette='Set2', alpha=0.7)
plt.title('Budget vs Risk Score by Success')
plt.xlabel('Budget (thousand $)')
plt.ylabel('Risk Score')
plt.legend(title='Success')
plt.show()

# Bar plot: Average duration by Complexity
plt.figure(figsize=(6,4))
sns.barplot(x='complexity_level', y='duration_days', data=df, ci=None)
plt.title('Average Duration by Complexity Level')
plt.xlabel('Complexity Level')
plt.ylabel('Average Duration (days)')
plt.show()

# Heatmap: Correlation matrix of numeric variables
plt.figure(figsize=(8,6))
numeric_cols = ['duration_days','budget_k','team_size','risk_score','vendor_count','deliverables_count']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.show()


In [ ]:
# Encode categorical variable complexity_level
df_encoded = df.copy()
df_encoded['complexity_level'] = df_encoded['complexity_level'].map({'Low':0, 'Medium':1, 'High':2})

# Feature matrix and target
y = df_encoded['success']
X = df_encoded[['duration_days','budget_k','team_size','risk_score','vendor_count','deliverables_count','complexity_level']]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train logistic regression model
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Predictions and evaluation
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Classification accuracy: {acc:.2f}")
print('Confusion Matrix:')
print(cm)


In [ ]:
# Regression model to predict project duration (in days)
y_reg = df_encoded['duration_days']
X_reg = df_encoded[['budget_k','team_size','risk_score','vendor_count','deliverables_count','complexity_level']]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

lin_reg = LinearRegression()
lin_reg.fit(X_train_reg, y_train_reg)

# Predictions
y_pred_reg = lin_reg.predict(X_test_reg)

# Evaluation
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"Mean Absolute Error: {mae:.2f}")
print(f"R2 Score: {r2:.2f}")
